In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logomaker

In [ ]:
h5_file = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/tfmodisco/counts/peaks_counts_modisco_results.h5"
pattern_group = "pos_patterns"
pattern_name = "pattern_0"

In [ ]:
trim_threshold = 0.3
flank = 4

In [ ]:
with h5py.File(h5_file, "r") as f:
    cwm = np.array(
        f[pattern_group][pattern_name]["contrib_scores"]
    )

In [ ]:
# cwm shape:
# motif_length × 4
# columns = A, C, G, T
print("Original shape:", cwm.shape)

In [ ]:
# 每个位置的总绝对贡献
position_score = np.sum(
    np.abs(cwm),
    axis=1
)
print(position_score)

In [ ]:
# 相对阈值：最大位置贡献的30%
threshold_value = (
    np.max(position_score) *
    trim_threshold
)
print(threshold_value)

In [ ]:
# 找到所有通过阈值的位置
pass_inds = np.where(
    position_score >= threshold_value
)[0]
if len(pass_inds) == 0:
    raise ValueError("No motif positions passed the trim threshold.")
print(pass_inds)

In [ ]:
# 高贡献核心边界
core_start = np.min(pass_inds)
core_end = np.max(pass_inds)
print(core_start,core_end)

In [ ]:
# report显示时额外增加左右各4 bp
plot_start = max(
    core_start - flank,
    0
)

plot_end = min(
    core_end + flank + 1,
    len(position_score)
)


trimmed_cwm = cwm[
    plot_start:plot_end
]

print("Maximum position score:", np.max(position_score))
print("Threshold:", threshold_value)
print(
    "Core, 0-based:",
    core_start,
    core_end
)

print(
    "Displayed range, 0-based:",
    plot_start,
    plot_end - 1
)

print(
    "Trimmed shape:",
    trimmed_cwm.shape
)


In [ ]:
# 画和report类似的CWM logo
logo_df = pd.DataFrame(
    trimmed_cwm,
    columns=["A", "C", "G", "T"]
)

fig, ax = plt.subplots(
    figsize=(max(5, len(logo_df) * 0.25), 2.5)
)

logomaker.Logo(
    logo_df,
    ax=ax,
    color_scheme="classic"
)

ax.axhline(
    0,
    linewidth=0.8,
    color="black"
)

ax.set_xlabel("Position")
ax.set_ylabel("Contribution score")

ax.set_xticks(range(len(logo_df)))
ax.set_xticklabels(
    range(plot_start + 1, plot_end + 1)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
#Modisco report 的核心规则
#position_score = abs(A) + abs(C) + abs(G) + abs(T)
#threshold = max(position_score) * trim_threshold
#确定核心区域后在左右各保留 flank=4 bp

In [ ]:
import os
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logomaker


def plot_modisco_cwm_logo(
    h5_file,
    pattern_name="pattern_0",
    pattern_group="pos_patterns",
    dataset_name="contrib_scores",
    output_pdf=None,
    trim_threshold=0.3,
    flank=4,
    strand="forward",
    edge_mode="clip",
    figsize=None,
    title=None,
    show_xticks=False,
    color_scheme="classic",
    shade_below=0.5,
    fade_below=0.5,
    show=True
):
    """
    读取 TF-MoDISco pattern 的 contribution matrix，
    按照 Modisco report 的规则进行 trim，并绘制 CWM logo。

    Parameters
    ----------
    h5_file : str
        TF-MoDISco 结果 HDF5 文件路径，例如：
        modisco_results_profile_scores.h5

    pattern_name : str, default "pattern_0"
        pattern 名称，例如：
        pattern_0、pattern_1、pattern_2。

    pattern_group : str, default "pos_patterns"
        motif 所在分组，可选：
        "pos_patterns"
        "neg_patterns"

    dataset_name : str, default "contrib_scores"
        HDF5 中使用的数据集。
        Modisco report 的 CWM logo 通常使用 contrib_scores。

    output_pdf : str or None
        PDF 输出路径。
        如果为 None，则不保存 PDF。

    trim_threshold : float, default 0.3
        相对于最大 position score 的阈值。

        每个位置的 score 为：
        abs(A) + abs(C) + abs(G) + abs(T)

        保留 score >= max(score) * trim_threshold 的核心区域。

    flank : int, default 4
        在核心区域左右额外保留的碱基数。

    strand : {"forward", "reverse", "both"}
        forward：
            绘制正向 motif。

        reverse：
            绘制 reverse-complement motif。

        both：
            同时绘制正向和反向互补 motif。

    edge_mode : {"clip", "shift"}, default "clip"
        核心靠近矩阵边缘时的处理方式。

        clip：
            与 Modisco report 的逻辑一致。
            超出矩阵边界的 flank 直接截断。

        shift：
            尽可能保持 core_length + 2*flank 的总长度。
            如果左侧不够，则把缺少的长度补到右侧；
            如果右侧不够，则补到左侧。

    figsize : tuple or None
        图尺寸。如果为 None，根据裁剪后 motif 长度自动设置。

    title : str or None
        图标题。如果为 None，使用 pattern_name。

    show_xticks : bool, default False
        是否显示原始 motif position。

    color_scheme : str or dict
        Logomaker 的碱基颜色方案。

    shade_below : float
        负贡献字母的阴影程度。

    fade_below : float
        负贡献字母的透明程度。

    show : bool, default True
        是否在 Jupyter 中显示图。

    Returns
    -------
    result : dict
        包含：

        figure
        axes
        metadata
        original_matrix
        output_pdf
    """

    # ------------------------------------------------------------
    # 1. 参数检查
    # ------------------------------------------------------------
    h5_file = Path(h5_file)

    if not h5_file.exists():
        raise FileNotFoundError(
            f"HDF5 file does not exist:\n{h5_file}"
        )

    if not 0 <= trim_threshold <= 1:
        raise ValueError(
            "trim_threshold should be between 0 and 1."
        )

    if flank < 0:
        raise ValueError(
            "flank must be >= 0."
        )

    if strand not in {"forward", "reverse", "both"}:
        raise ValueError(
            "strand must be 'forward', 'reverse', or 'both'."
        )

    if edge_mode not in {"clip", "shift"}:
        raise ValueError(
            "edge_mode must be 'clip' or 'shift'."
        )

    # ------------------------------------------------------------
    # 2. 从 HDF5 读取 contribution matrix
    # ------------------------------------------------------------
    dataset_path = (
        f"{pattern_group}/"
        f"{pattern_name}/"
        f"{dataset_name}"
    )

    with h5py.File(h5_file, "r") as f:

        if dataset_path not in f:
            available_patterns = []

            if pattern_group in f:
                available_patterns = list(
                    f[pattern_group].keys()
                )

            raise KeyError(
                f"Dataset not found:\n"
                f"{dataset_path}\n\n"
                f"Available patterns in {pattern_group}:\n"
                f"{available_patterns}"
            )

        cwm = np.asarray(
            f[dataset_path],
            dtype=float
        )

    # ------------------------------------------------------------
    # 3. 检查矩阵方向
    #
    # 标准形式：
    # 行 = motif position
    # 列 = A/C/G/T
    # shape = motif_length × 4
    # ------------------------------------------------------------
    if cwm.ndim != 2:
        raise ValueError(
            f"Expected a two-dimensional matrix, "
            f"but obtained shape {cwm.shape}."
        )

    if cwm.shape[1] == 4:
        pass

    elif cwm.shape[0] == 4:
        print(
            "The matrix shape is 4 × motif_length; "
            "it has been transposed automatically."
        )
        cwm = cwm.T

    else:
        raise ValueError(
            "The contribution matrix must have four nucleotide "
            "channels. Expected shape motif_length × 4 or "
            f"4 × motif_length, but obtained {cwm.shape}."
        )

    if not np.isfinite(cwm).all():
        raise ValueError(
            "The matrix contains NaN or infinite values."
        )

    motif_length = cwm.shape[0]

    if motif_length == 0:
        raise ValueError(
            "The contribution matrix is empty."
        )

    # ------------------------------------------------------------
    # 4. 确定需要绘制的方向
    #
    # 原始通道顺序：
    # A C G T
    #
    # 两个轴同时反转：
    # position 反转
    # A/C/G/T -> T/G/C/A
    #
    # 即得到 reverse complement。
    # ------------------------------------------------------------
    if strand == "forward":

        matrices_to_plot = {
            "forward": cwm
        }

    elif strand == "reverse":

        matrices_to_plot = {
            "reverse": cwm[::-1, ::-1]
        }

    else:

        matrices_to_plot = {
            "forward": cwm,
            "reverse": cwm[::-1, ::-1]
        }

    # ------------------------------------------------------------
    # 5. 对每个方向计算 trim 区间
    # ------------------------------------------------------------
    plot_results = {}

    for strand_name, strand_matrix in matrices_to_plot.items():

        # 每个 position 的总绝对贡献
        position_score = np.sum(
            np.abs(strand_matrix),
            axis=1
        )

        max_position_score = np.max(
            position_score
        )

        threshold_value = (
            max_position_score *
            trim_threshold
        )

        # 如果整个矩阵都是 0，则保留全部位置
        if max_position_score == 0:

            pass_indices = np.arange(
                strand_matrix.shape[0]
            )

        else:

            pass_indices = np.where(
                position_score >= threshold_value
            )[0]

        if len(pass_indices) == 0:
            raise ValueError(
                f"No positions passed the trim threshold "
                f"for {strand_name} strand."
            )

        # 核心区域
        core_start = int(
            np.min(pass_indices)
        )

        core_end = int(
            np.max(pass_indices)
        )

        # --------------------------------------------------------
        # 初始显示范围：核心区域左右各增加 flank
        #
        # plot_end 是 Python 左闭右开的结束坐标
        # --------------------------------------------------------
        plot_start = max(
            core_start - flank,
            0
        )

        plot_end = min(
            core_end + flank + 1,
            strand_matrix.shape[0]
        )

        # --------------------------------------------------------
        # 6. 处理核心靠近矩阵边缘的情况
        # --------------------------------------------------------
        if edge_mode == "shift":

            core_length = (
                core_end -
                core_start +
                1
            )

            target_length = min(
                core_length + 2 * flank,
                strand_matrix.shape[0]
            )

            current_length = (
                plot_end -
                plot_start
            )

            missing_length = (
                target_length -
                current_length
            )

            # 左侧到达边缘：
            # 将缺少的 flank 尽可能补到右侧
            if missing_length > 0 and plot_start == 0:

                plot_end = min(
                    strand_matrix.shape[0],
                    plot_end + missing_length
                )

            current_length = (
                plot_end -
                plot_start
            )

            missing_length = (
                target_length -
                current_length
            )

            # 右侧到达边缘：
            # 将缺少的 flank 尽可能补到左侧
            if (
                missing_length > 0
                and plot_end == strand_matrix.shape[0]
            ):

                plot_start = max(
                    0,
                    plot_start - missing_length
                )

        trimmed_matrix = strand_matrix[
            plot_start:plot_end,
            :
        ]

        plot_results[strand_name] = {
            "matrix": trimmed_matrix,
            "position_score": position_score,
            "threshold_value": threshold_value,
            "max_position_score": max_position_score,
            "core_start": core_start,
            "core_end": core_end,
            "plot_start": plot_start,
            "plot_end": plot_end,
            "left_flank_kept": core_start - plot_start,
            "right_flank_kept": plot_end - core_end - 1
        }

    # ------------------------------------------------------------
    # 7. 创建图形
    # ------------------------------------------------------------
    number_of_panels = len(
        plot_results
    )

    longest_trimmed_length = max(
        value["matrix"].shape[0]
        for value in plot_results.values()
    )

    if figsize is None:

        figure_width = max(
            5,
            longest_trimmed_length * 0.30
        )

        figure_height = (
            2.5 * number_of_panels
        )

        figsize = (
            figure_width,
            figure_height
        )

    fig, axes = plt.subplots(
        nrows=number_of_panels,
        ncols=1,
        figsize=figsize,
        squeeze=False
    )

    axes = axes[:, 0]

    base_title = (pattern_name if title is None else title)

    # ------------------------------------------------------------
    # 8. 绘制每个方向的 logo
    # ------------------------------------------------------------
    for ax, (strand_name,result) in zip(axes,plot_results.items()):

        trimmed_matrix = result["matrix"]

        logo_df = pd.DataFrame(
            trimmed_matrix,
            columns=["A","C","G","T"]
        )

        logomaker.Logo(
            logo_df,
            ax=ax,
            color_scheme=color_scheme,
            shade_below=shade_below,
            fade_below=fade_below,
            flip_below=True #用于控制 零轴以下的负值字母是否上下翻转
        )

        # 零线
        ax.axhline(
            y=0,
            color="black",
            linewidth=0.7
        )

        # --------------------------------------------------------
        # 根据正负值的堆叠高度设置 y 轴
        # --------------------------------------------------------
        #np.clip()用来把数组中的数值限制在指定范围内：
        #np.clip(array,a_min,a_max)小于 a_min 的值变成 a_min；大于 a_max 的值变成 a_max
        #例如某个位置：A =  0.02 C = -0.01 G =  0.03 T = -0.02
        #np.clip(values, a_min=0, a_max=None)后变成A = 0.02 C = 0 G = 0.03 T = 0
        #然后求和positive_height = 0.02 + 0.03 = 0.05代表该位置在 logo 中零轴上方的总堆叠高度
        positive_height = np.sum(
            np.clip(
                trimmed_matrix,
                a_min=0,
                a_max=None
            ),
            axis=1
        )

        negative_height = np.sum(
            np.clip(
                trimmed_matrix,
                a_min=None,
                a_max=0
            ),
            axis=1
        )

        max_positive = max(
            float(np.max(positive_height)),
            0
        )

        min_negative = min(
            float(np.min(negative_height)),
            0
        )

        total_range = (max_positive -min_negative)

        if total_range == 0:#当整个trimed矩阵全部为0时
            total_range = 1

        padding = total_range * 0.08

        y_min = (
            min_negative - padding
            if min_negative < 0
            else -padding
        )

        y_max = (
            max_positive + padding
            if max_positive > 0
            else padding
        )

        ax.set_ylim(y_min,y_max)

        # 标题
        if number_of_panels == 1:
            panel_title = base_title
        else:
            panel_title = (
                f"{base_title} "
                f"({strand_name})"
            )

        ax.set_title(
            panel_title,
            fontsize=13
        )

        ax.set_ylabel(
            "Contribution score",
            fontsize=10
        )

        # --------------------------------------------------------
        # x 轴位置
        # --------------------------------------------------------
        if show_xticks:

            original_positions = np.arange(
                result["plot_start"] + 1,#将坐标转换成1-based
                result["plot_end"] + 1#为了包含最后一个位置
            )

            ax.set_xticks(
                np.arange(
                    trimmed_matrix.shape[0]
                )
            )

            ax.set_xticklabels(
                original_positions,
                fontsize=8
            )

            ax.set_xlabel(
                "Position in original TF-MoDISco pattern"
            )

        else:

            ax.set_xticks([])

        # 去除不必要边框
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.tight_layout()

    # ------------------------------------------------------------
    # 9. 保存 PDF
    # ------------------------------------------------------------
    saved_pdf = None

    if output_pdf is not None:

        output_pdf = Path(output_pdf)

        # 如果没有 .pdf 后缀，自动添加
        if output_pdf.suffix.lower() != ".pdf":
            output_pdf = output_pdf.with_suffix(
                ".pdf"
            )

        output_pdf.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        fig.savefig(
            output_pdf,
            format="pdf",
            bbox_inches="tight"
        )

        saved_pdf = str(
            output_pdf.resolve()#相对路径准换成绝对路径，为了后面把实际保存位置完整地打印和返回
        )

        print(
            f"PDF saved to:\n{saved_pdf}"
        )

    # ------------------------------------------------------------
    # 10. 输出裁剪信息
    # ------------------------------------------------------------
    for strand_name, result in plot_results.items():

        print(
            "\n"
            "----------------------------------------"
        )

        print(
            f"Pattern: {pattern_name}"
        )

        print(
            f"Strand: {strand_name}"
        )

        print(
            f"Original motif length: {motif_length}"
        )

        print(
            "Maximum position score: "
            f"{result['max_position_score']:.6g}"
        )

        print(
            "Trim threshold value: "
            f"{result['threshold_value']:.6g}"
        )

        print(
            "Core range, 0-based inclusive: "
            f"{result['core_start']} - "
            f"{result['core_end']}"
        )

        print(
            "Displayed range, 0-based inclusive: "
            f"{result['plot_start']} - "
            f"{result['plot_end'] - 1}"
        )

        print(
            "Displayed range, 1-based inclusive: "
            f"{result['plot_start'] + 1} - "
            f"{result['plot_end']}"
        )

        print(
            "Actual left/right flank kept: "
            f"{result['left_flank_kept']} / "
            f"{result['right_flank_kept']} bp"
        )

        print(
            "Trimmed motif length: "
            f"{result['matrix'].shape[0]}"
        )

    if show:
        plt.show()

    else:
        plt.close(fig)

    return {
        "figure": fig,
        "axes": axes,
        "metadata": plot_results,
        "original_matrix": cwm,
        "output_pdf": saved_pdf
    }

In [ ]:
# display all outputs in a cell
get_ipython().ast_node_interactivity = 'all'

In [ ]:
h5_file = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/tfmodisco/counts/peaks_counts_modisco_results.h5"
output_dir = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/tfmodisco/counts/modisco_cwm_logos"

os.makedirs(
    output_dir,
    exist_ok=True
)

for i in range(10):

    pattern_name = f"pattern_{i}"

    plot_modisco_cwm_logo(
        h5_file=h5_file,
        pattern_group="pos_patterns",
        pattern_name=pattern_name,
        output_pdf=os.path.join(
            output_dir,
            f"{pattern_name}_cwm_logo.pdf"
        ),
        trim_threshold=0.3,
        flank=4,
        strand="forward",
        edge_mode="clip",
        show=True
    )

In [ ]:
h5_file = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/output/predict/C8.1/tfmodisco/C8.1_counts_modisco_results.h5"
output_dir = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/output/predict/C8.1/tfmodisco/modisco_cwm_logos"

os.makedirs(
    output_dir,
    exist_ok=True
)

for i in range(10):

    pattern_name = f"pattern_{i}"

    plot_modisco_cwm_logo(
        h5_file=h5_file,
        pattern_group="pos_patterns",
        pattern_name=pattern_name,
        output_pdf=os.path.join(
            output_dir,
            f"{pattern_name}_cwm_logo.pdf"
        ),
        trim_threshold=0.3,
        flank=4,
        strand="forward",
        edge_mode="clip",
        show=True
    )

In [ ]:
h5_file = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C10Hepatocytes/output/predict/C10.1/tfmodisco/C10.1_counts_modisco_results.h5"
output_dir = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C10Hepatocytes/output/predict/C10.1/tfmodisco/modisco_cwm_logos"

os.makedirs(
    output_dir,
    exist_ok=True
)

for i in range(10):

    pattern_name = f"pattern_{i}"

    plot_modisco_cwm_logo(
        h5_file=h5_file,
        pattern_group="pos_patterns",
        pattern_name=pattern_name,
        output_pdf=os.path.join(
            output_dir,
            f"{pattern_name}_cwm_logo.pdf"
        ),
        trim_threshold=0.3,
        flank=4,
        strand="forward",
        edge_mode="clip",
        show=True
    )

In [ ]:
h5_file = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C19Olfactory_sensory_neurons/output/predict/C19.1/tfmodisco/C19.1_counts_modisco_results.h5"
output_dir = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C19Olfactory_sensory_neurons/output/predict/C19.1/tfmodisco/modisco_cwm_logos"

os.makedirs(
    output_dir,
    exist_ok=True
)

for i in range(10):

    pattern_name = f"pattern_{i}"

    plot_modisco_cwm_logo(
        h5_file=h5_file,
        pattern_group="pos_patterns",
        pattern_name=pattern_name,
        output_pdf=os.path.join(
            output_dir,
            f"{pattern_name}_cwm_logo.pdf"
        ),
        trim_threshold=0.3,
        flank=4,
        strand="forward",
        edge_mode="clip",
        show=True
    )

In [ ]:
h5_file = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C24white_blood_cells/output/predict/C24.3/tfmodisco/C24.3_counts_modisco_results.h5"
output_dir = "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C24white_blood_cells/output/predict/C24.3/tfmodisco/modisco_cwm_logos"

os.makedirs(
    output_dir,
    exist_ok=True
)

for i in range(10):

    pattern_name = f"pattern_{i}"

    plot_modisco_cwm_logo(
        h5_file=h5_file,
        pattern_group="pos_patterns",
        pattern_name=pattern_name,
        output_pdf=os.path.join(
            output_dir,
            f"{pattern_name}_cwm_logo.pdf"
        ),
        trim_threshold=0.3,
        flank=4,
        strand="forward",
        edge_mode="clip",
        show=True
    )